# Day 168 — RAG + LangChain + Groq Capstone
## Month 9 Flagship: Build a Production-Grade QA System Over Client Review Data
**Dataset:** ReviewPulse India (600 rows, seed=155) | **Environment:** Google Colab

---
> **Business framing:** A freelance platform wants to let account managers query their client review database in natural language — *'Which freelancers have communication issues?'* or *'What drives re-hire decisions?'* — without writing SQL. You are building that system using RAG (Retrieval-Augmented Generation).

---
### Architecture
```
ReviewPulse Reviews
       │
   [LangChain]
       │
   Document Loader → Text Splitter → Embeddings (all-MiniLM-L6-v2)
       │                                       │
   FAISS Vector Store ←──────────────────────┘
       │
   Retriever (k=3)
       │
   Prompt Template
       │
   Groq LLM (llama-3.1-8b-instant)
       │
   Answer + RAGAS Evaluation
```

**Stack:** `langchain` · `langchain-community` · `langchain-groq` · `sentence-transformers` · `faiss-cpu` · `ragas` · `datasets`

## ⚠️ RAW DATA CELL — DO NOT MODIFY

In [4]:
# ════════════════════════════════════════════════════════
# RAW DATA — DO NOT MODIFY THIS CELL
# ════════════════════════════════════════════════════════
import numpy as np
import pandas as pd

np.random.seed(155)
n = 600
sentiments = np.random.choice(['positive','negative','neutral'], n, p=[0.25,0.45,0.30])
ratings = np.where(sentiments=='positive', np.random.randint(4,6,n),
          np.where(sentiments=='negative', np.random.randint(1,3,n),
                   np.random.randint(2,5,n)))
hired_again = np.where(
    (sentiments=='positive') & (ratings>=4), np.random.choice([0,1],n,p=[0.15,0.85]),
    np.where((sentiments=='negative') & (ratings<=2), np.random.choice([0,1],n,p=[0.90,0.10]),
             np.random.choice([0,1],n,p=[0.70,0.30]))
)
pos_t = ['Excellent work, very professional and delivered on time.',
         'Outstanding quality, highly recommend this freelancer.',
         'Great communication and superb results, will hire again.',
         'Fantastic job, exceeded all expectations completely.',
         'Very skilled professional, perfect delivery every time.']
neg_t = ['Poor quality work, missed deadlines and ignored feedback.',
         'Very disappointing, did not meet requirements at all.',
         'Terrible communication, work was substandard and late.',
         'Would not recommend, very unprofessional behavior shown.',
         'Bad experience overall, refund was requested immediately.']
neu_t = ['Work was okay, met basic requirements nothing special.',
         'Average performance, some delays but completed eventually.',
         'Decent work quality, communication could be improved.',
         'Satisfactory output, would consider hiring again maybe.',
         'Meets expectations, professional but not exceptional work.']

def gen_review(s, seed_offset):
    np.random.seed(155 + seed_offset)
    templates = pos_t if s=='positive' else neg_t if s=='negative' else neu_t
    base = np.random.choice(templates)
    words = base.split()
    np.random.shuffle(words)
    return ' '.join(words).capitalize() + ('!' if s=='positive' else '.')

reviews_list = [gen_review(s, i) for i, s in enumerate(sentiments)]
df_raw = pd.DataFrame({
    'freelancer_id': [f'FL{str(i+1).zfill(4)}' for i in range(n)],
    'review_text': reviews_list,
    'rating': ratings,
    'sentiment': sentiments,
    'hired_again': hired_again
})
print(f'Dataset loaded: {df_raw.shape}')
print(df_raw['sentiment'].value_counts())
df_raw.head(3)

Dataset loaded: (600, 5)
sentiment
negative    267
neutral     180
positive    153
Name: count, dtype: int64


,freelancer_id,review_text,rating,sentiment,hired_again
0,FL0001,"Requested refund immediately. overall, was bad...",1,negative,0
1,FL0002,"Work. meets expectations, exceptional but prof...",2,neutral,0
2,FL0003,"Delivered time. excellent and work, on very pr...",5,positive,1


## Section 1 — Install & Setup
Run once per Colab session. Groq API is **free** — get your key at https://console.groq.com

In [5]:
# Install required packages
!pip install -q langchain langchain-community langchain-groq \
               sentence-transformers faiss-cpu \
               ragas datasets
print('Installation complete.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [6]:
import os
from google.colab import userdata

# ── Set Groq API key ──
# Option A: Add GROQ_API_KEY in Colab Secrets (Colab → Runtime → Secrets)
# Option B: paste directly (not recommended for shared notebooks)
try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('Groq API key loaded from Colab Secrets.')
except:
    os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'  # replace if needed
    print('Using fallback key — add to Colab Secrets for production.')

Groq API key loaded from Colab Secrets.


---
## Concept Notes
### RAG — Why it exists
LLMs are trained on general text up to a cutoff date. They hallucinate when asked about your private data. RAG (Retrieval-Augmented Generation) solves this by:
1. **Indexing** your documents as vector embeddings
2. **Retrieving** the top-k most relevant chunks for each query
3. **Augmenting** the LLM prompt with those chunks as context
4. **Generating** a grounded answer — the LLM can only say what the retrieved context supports

### LangChain — What it orchestrates
LangChain chains together: `Loader → Splitter → Embedder → VectorStore → Retriever → Prompt → LLM → Output`. Each component is swappable — swap Groq for OpenAI, FAISS for Chroma, all-MiniLM for OpenAI embeddings.

### RAGAS — How to evaluate RAG
| Metric | What it measures | Good score |
|--------|-----------------|------------|
| `faithfulness` | Are all answer claims in the context? | >0.8 |
| `answer_relevancy` | Does the answer address the question? | >0.8 |
| `context_precision` | Is retrieved context actually relevant? | >0.7 |
| `context_recall` | Does context cover the ground truth? | >0.7 |

### Key vocabulary
- **Document**: one unit of text fed to the system (here: one review row)
- **Chunk**: a smaller split of a document (here: reviews are short → 1 chunk each)
- **Embedding**: 384-dim float vector capturing semantic meaning
- **FAISS**: Facebook AI Similarity Search — fast nearest-neighbour lookup in embedding space
- **Retriever**: returns top-k documents whose embeddings are closest to the query embedding

---
## T1 — Document Preparation + LangChain Document Objects
**[15 points]**

### Task
Convert `df_raw` into LangChain `Document` objects — one per review — with structured metadata.
Then apply `RecursiveCharacterTextSplitter` (chunk_size=200, chunk_overlap=20) and confirm chunk count.

### Skeleton

In [7]:
# Part: T1 - Document Preparation
# Goal: Create Document objects with metadata and split into chunks.
# Method: Use Document from langchain_core.documents, RecursiveCharacterTextSplitter.

# ── IMPORTS ──
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── TASK 1A: Create Document objects ──
documents = [
    Document(
        page_content=row['review_text'],
        metadata={
            'freelancer_id': row['freelancer_id'],
            'rating': int(row['rating']),
            'sentiment': row['sentiment'],
            'hired_again': int(row['hired_again'])
        }
    )
    for _, row in df_raw.iterrows()
]

print(f'Documents created: {len(documents)}')
print(f'Sample document:')
print(f'  content: {documents[0].page_content}')
print(f'  metadata: {documents[0].metadata}')

# ── TASK 1B: Apply text splitter ──
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(documents)
print(f'\nChunks after splitting: {len(chunks)}')
print(f'Avg chunk length: {sum(len(c.page_content) for c in chunks)/len(chunks):.1f} chars')

# ── TASK 1C: NRA Insight ──
nra_t1 = """
Number: 600 chunks
Reason: Each review is a short paragraph (avg 60-80 chars), well below the 200-character chunk size, so the splitter does not split any document; each document becomes exactly one chunk.
Action: For longer client documents (e.g., full project descriptions), I would increase chunk_size to 512 or 1000 to capture more context per chunk, and add a 10% overlap to preserve continuity.
"""
print(nra_t1)

Documents created: 600
Sample document:
  content: Requested refund immediately. overall, was bad experience.
  metadata: {'freelancer_id': 'FL0001', 'rating': 1, 'sentiment': 'negative', 'hired_again': 0}

Chunks after splitting: 600
Avg chunk length: 56.3 chars

Number: 600 chunks
Reason: Each review is a short paragraph (avg 60-80 chars), well below the 200-character chunk size, so the splitter does not split any document; each document becomes exactly one chunk.
Action: For longer client documents (e.g., full project descriptions), I would increase chunk_size to 512 or 1000 to capture more context per chunk, and add a 10% overlap to preserve continuity.



---
## T2 — Embeddings + FAISS Vector Store
**[15 points]**

### Task
Build a FAISS vector store from the chunks using `all-MiniLM-L6-v2`. Run a similarity search to verify retrieval is working. Then inspect embedding dimensions.

### Skeleton

In [8]:
# Part: T2 - Embeddings and FAISS Vector Store
# Goal: Build FAISS index with sentence-transformers embeddings, verify retrieval.
# Method: Use HuggingFaceEmbeddings, FAISS.from_documents, similarity_search.

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ── TASK 2A: Load embedding model ──
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)
print('Embedding model loaded.')

# Test embedding dimensions
test_vec = embedding_model.embed_query('test')
print(f'Embedding dimensions: {len(test_vec)}')

# ── TASK 2B: Build FAISS vector store ──
vectorstore = FAISS.from_documents(chunks, embedding_model)
print(f'\nFAISS index built. Total vectors: {vectorstore.index.ntotal}')

# ── TASK 2C: Verify retrieval with a test query ──
test_query = 'freelancers with communication problems'
results = vectorstore.similarity_search(test_query, k=3)
print(f'\nTop-3 results for: "{test_query}"')
for i, doc in enumerate(results):
    print(f'\n[{i+1}] Sentiment: {doc.metadata["sentiment"]} | Rating: {doc.metadata["rating"]}')
    print(f'     Text: {doc.page_content}')

# ── TASK 2D: NRA Insight ──
nra_t2 = """
Number: 3/3 retrieved docs are positive sentiment (fraction = 1.0)
Reason: The query "freelancers with communication problems" centers on the domain of freelancer quality and professionalism. The all-MiniLM embedding model finds nearest neighbours that are top‑scoring positive reviews about freelancers, because the query's core topic (freelancer quality) matches the positive review content more strongly than the sentiment intent (problems). This is a topic‑similarity effect, not a semantic closeness between "problems" and words like "recommend".
Action: In production, add a metadata filter to restrict retrieval to negative/neutral reviews when querying about problems, or use a classifier to pre‑filter sentiment before retrieval.
"""
print(nra_t2)

/tmp/ipykernel_1091/2446816109.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_1091/2446816109.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your set

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.
Embedding dimensions: 384

FAISS index built. Total vectors: 600

Top-3 results for: "freelancers with communication problems"

[1] Sentiment: positive | Rating: 4
     Text: Freelancer. recommend outstanding this quality, highly!

[2] Sentiment: positive | Rating: 5
     Text: Freelancer. outstanding quality, this recommend highly!

[3] Sentiment: positive | Rating: 4
     Text: Freelancer. highly quality, outstanding recommend this!

Number: 3/3 retrieved docs are positive sentiment (fraction = 1.0)
Reason: The query "freelancers with communication problems" centers on the domain of freelancer quality and professionalism. The all-MiniLM embedding model finds nearest neighbours that are top‑scoring positive reviews about freelancers, because the query's core topic (freelancer quality) matches the positive review content more strongly than the sentiment intent (problems). This is a topic‑similarity effect, not a semantic closeness between "problems" and words li

---
## T3 — LangChain RAG Chain with Groq LLM
**[20 points]**

### Task
Build the full RAG chain: retriever → prompt → Groq LLM → answer. Run **4 business queries** against the system. Print retrieved context + final answer for each.

### Skeleton

In [9]:
# Part: T3 - RAG Chain with Groq LLM (Runnable API)
# Goal: Build and run full RAG pipeline with 4 business queries.
# Method: Use langchain_core runnables for reliable retrieval + generation.

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── TASK 3A: Initialize Groq LLM ──
llm = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0,
    api_key=os.environ['GROQ_API_KEY']
)
print('Groq LLM initialized.')

# ── TASK 3B: Build RAG prompt template ──
rag_prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template="""
You are a freelance platform analyst. Answer the question using ONLY the review context provided below.
If the context does not contain enough information, say 'Insufficient context'.
Always mention the sentiment labels (positive/negative/neutral) present in the context.

Context:
{context}

Question: {question}

Answer:"""
)

# ── TASK 3C: Build RAG chain using runnables ──
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print('RAG chain built.')

# Helper to get answer + source docs
def rag_with_sources(query):
    docs = retriever.invoke(query)
    answer = rag_chain.invoke(query)
    return {"result": answer, "source_documents": docs}

# ── TASK 3D: Run 4 business queries ──
queries = [
    'What are the most common complaints about freelancer communication?',
    'Which types of reviews lead to the client hiring again?',
    'What characterises high-rating freelancer reviews?',
    'What are the warning signs in reviews that predict a client will not hire again?'
]

answers = {}
for q in queries:
    print(f'\n{"="*60}')
    print(f'QUERY: {q}')
    result = rag_with_sources(q)
    answers[q] = result
    print(f'\nANSWER:\n{result["result"]}')
    print(f'\nSOURCE DOCS ({len(result["source_documents"])}):')
    for doc in result['source_documents']:
        print(f'  [{doc.metadata["sentiment"]} | rating={doc.metadata["rating"]}] {doc.page_content[:60]}...')

# ── TASK 3E: NRA Insight ──
nra_t3 = """
Query chosen: 'What are the most common complaints about freelancer communication?'
Number: 2 out of 3 retrieved docs are negative (fraction = 0.67); the LLM response lists "ignored feedback" and "missed deadlines".
Reason: The retriever returned 2 negative and 1 positive review. The positive review does not mention complaints, so the LLM correctly ignored it and used only the negative ones. This demonstrates that the model follows the instruction to use only the provided context.
Action: To get a more comprehensive list, increase k to 5 and filter by sentiment='negative' for complaints queries to avoid dilution from irrelevant positive reviews.
"""
print(nra_t3)

Groq LLM initialized.
RAG chain built.

QUERY: What are the most common complaints about freelancer communication?

ANSWER:
Based on the provided context, the most common complaints about freelancer communication are:

1. Ignored feedback
2. Missed deadlines

Sentiment labels: Negative (both contexts)

SOURCE DOCS (3):
  [negative | rating=1] Work, quality feedback. poor ignored and missed deadlines....
  [positive | rating=4] Freelancer. recommend outstanding this quality, highly!...
  [negative | rating=2] Work, poor quality deadlines and missed ignored feedback.....

QUERY: Which types of reviews lead to the client hiring again?

ANSWER:
Insufficient context.

SOURCE DOCS (3):
  [negative | rating=1] Work, quality feedback. poor ignored and missed deadlines....
  [negative | rating=1] Quality missed feedback. poor ignored work, deadlines and....
  [negative | rating=2] Feedback. work, ignored poor and missed quality deadlines....

QUERY: What characterises high-rating freelancer rev

---
## T4 — Metadata Filtering + MMR Retrieval
**[15 points]**

### Task
LangChain FAISS supports two advanced retrieval modes:
1. **Metadata filtering**: retrieve only docs matching a filter (e.g., only negative-sentiment reviews)
2. **MMR (Maximal Marginal Relevance)**: retrieve diverse docs, not just the nearest neighbours

Run both modes, compare results against the baseline k=3 similarity search.

### Skeleton

In [10]:
# Part: T4 - Metadata Filtering + MMR Retrieval
# Goal: Demonstrate filtered and MMR retrieval, compare to baseline.
# Method: as_retriever with filter and search_type='mmr'.

# ── TASK 4A: Metadata-filtered retrieval ──
filtered_retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k': 3,
        'filter': {'sentiment': 'negative'}
    }
)
filter_query = 'communication and deadline issues'
filtered_results = filtered_retriever.invoke(filter_query)
print(f'Filtered retrieval (negative only) for: "{filter_query}"')
for doc in filtered_results:
    print(f'  [{doc.metadata["sentiment"]} | hired={doc.metadata["hired_again"]}] {doc.page_content}')

# ── TASK 4B: MMR retrieval ──
mmr_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.5}
)
mmr_results = mmr_retriever.invoke(filter_query)
print(f'\nMMR retrieval for: "{filter_query}"')
for doc in mmr_results:
    print(f'  [{doc.metadata["sentiment"]} | hired={doc.metadata["hired_again"]}] {doc.page_content}')

# ── TASK 4C: Comparison table ──
baseline_results = vectorstore.similarity_search(filter_query, k=3)
comparison = pd.DataFrame({
    'Rank': [1, 2, 3],
    'Similarity (baseline)': [d.metadata['sentiment'] for d in baseline_results],
    'Filtered (negative)': [d.metadata['sentiment'] for d in filtered_results],
    'MMR': [d.metadata['sentiment'] for d in mmr_results]
})
print('\nRetrieval Mode Comparison:')
print(comparison.to_string(index=False))

# ── TASK 4D: NRA Insight ──
nra_t4 = """
Number: MMR and filtered retrieval returned the same 3 documents (only the order changed), so no diversity gain was observed (0 docs differ between MMR and baseline).
Reason: With lambda_mult=0.5, the relevance–diversity trade‑off did not introduce new documents because the top candidates were already too similar. The fetched candidates (fetch_k=10) likely contained only near‑duplicates of the same negative phrases.
Action: Increase `fetch_k` to a larger value (e.g., 20) to give MMR more candidates to choose from, and adjust `lambda_mult` **towards 0** (e.g., 0.2) to increase diversity, because lambda_mult=1.0 is pure relevance and 0.0 is pure diversity. For this dataset, metadata filtering is more effective than MMR to ensure diverse sentiment coverage.
"""
print(nra_t4)


Filtered retrieval (negative only) for: "communication and deadline issues"
  [negative | hired=1] Work, ignored and feedback. poor deadlines quality missed.
  [negative | hired=0] Poor deadlines quality ignored feedback. missed and work,.
  [negative | hired=0] Feedback. deadlines poor missed and ignored work, quality.

MMR retrieval for: "communication and deadline issues"
  [negative | hired=1] Work, ignored and feedback. poor deadlines quality missed.
  [negative | hired=0] Feedback. deadlines poor missed and ignored work, quality.
  [negative | hired=0] Poor deadlines quality ignored feedback. missed and work,.

Retrieval Mode Comparison:
 Rank Similarity (baseline) Filtered (negative)      MMR
    1              negative            negative negative
    2              negative            negative negative
    3              negative            negative negative

Number: MMR and filtered retrieval returned the same 3 documents (only the order changed), so no diversity gain was obs

---
## T5 — RAGAS Evaluation
**[15 points]**

### Task
Evaluate your RAG system using RAGAS on 4 QA pairs. Build an evaluation dataset with ground-truth answers, compute all 4 RAGAS metrics, and interpret results.

> **Note on ground truth:** For this exercise, ground truth answers are derived from the raw dataset statistics (precomputed). Use these numbers as your reference answers.

### Skeleton

In [11]:
!pip install -q google-cloud-aiplatform

In [12]:
# ── T5 — RAGAS Evaluation (Manual, Groq-based, with Robust Parsing) ──

import re
import pandas as pd

# 1. Define evaluation questions and ground truths
eval_questions = [
    'What types of issues appear in negative freelancer reviews?',
    'What characterises reviews where the client hired again?',
    'What is the approximate proportion of negative reviews in the dataset?',
    'What communication-related issues are mentioned in poor reviews?'
]

ground_truths = [
    'Negative reviews commonly mention missed deadlines, poor communication, substandard work quality, and unprofessional behavior.',
    'Positive reviews with ratings of 4 or 5 are strongly associated with repeat hiring, with an 85% rehire rate.',
    'Approximately 44.5% of reviews are negative, making it the largest sentiment category in the dataset.',
    'Poor reviews frequently cite terrible communication, ignored feedback, and delayed deliveries as the main complaints.'
]

# 2. Collect answers and contexts using your existing rag_with_sources
ragas_answers = []
ragas_contexts = []
for q in eval_questions:
    result = rag_with_sources(q)   # returns dict with 'result' and 'source_documents'
    ragas_answers.append(result['result'])
    ragas_contexts.append([doc.page_content for doc in result['source_documents']])
    print(f'\nQ: {q}')
    print(f'A: {result["result"][:120]}...')

# 3. Helper to extract a float from LLM response (robust)
def extract_float(text):
    # Find the first floating-point number in the text
    match = re.search(r'(\d+\.\d+)', text)
    if match:
        return float(match.group(1))
    # If no decimal, try integer
    match_int = re.search(r'(\d+)', text)
    if match_int:
        return float(match_int.group(1))
    return None

# 4. Define scoring functions with explicit "ONLY a number" instruction
def score_faithfulness(question, answer, context):
    prompt = f"""
You are an evaluator. Given a QUESTION, an ANSWER, and some CONTEXT, determine whether the ANSWER is fully supported by the CONTEXT.
Return ONLY a number between 0 and 1, where 1 means all claims in the answer are directly supported by the context, and 0 means none are supported.

QUESTION: {question}
ANSWER: {answer}
CONTEXT: {context}

Score (0 to 1). Return ONLY a number between 0 and 1, with no other text, explanation, or formatting:
"""
    response = llm.invoke(prompt)
    score = extract_float(response.content.strip())
    if score is not None:
        return max(0.0, min(1.0, score))
    return 0.0

def score_answer_relevancy(question, answer):
    prompt = f"""
You are an evaluator. Given a QUESTION and an ANSWER, assess how relevant the answer is to the question.
Return ONLY a number between 0 and 1, where 1 means the answer directly addresses the question, and 0 means it's completely irrelevant.

QUESTION: {question}
ANSWER: {answer}

Score (0 to 1). Return ONLY a number between 0 and 1, with no other text, explanation, or formatting:
"""
    response = llm.invoke(prompt)
    score = extract_float(response.content.strip())
    if score is not None:
        return max(0.0, min(1.0, score))
    return 0.0

def score_context_precision(question, context):
    prompt = f"""
You are an evaluator. Given a QUESTION and some CONTEXT, determine whether the CONTEXT contains information that is useful for answering the question.
Return ONLY a number between 0 and 1, where 1 means the context is highly relevant and contains necessary information, and 0 means it's completely irrelevant.

QUESTION: {question}
CONTEXT: {context}

Score (0 to 1). Return ONLY a number between 0 and 1, with no other text, explanation, or formatting:
"""
    response = llm.invoke(prompt)
    score = extract_float(response.content.strip())
    if score is not None:
        return max(0.0, min(1.0, score))
    return 0.0

def score_context_recall(ground_truth, context):
    prompt = f"""
You are an evaluator. Given a GROUND TRUTH answer and some CONTEXT, determine what fraction of the information in the ground truth is covered by the context.
Return ONLY a number between 0 and 1, where 1 means the context contains all information present in the ground truth, and 0 means it contains none.

GROUND TRUTH: {ground_truth}
CONTEXT: {context}

Score (0 to 1). Return ONLY a number between 0 and 1, with no other text, explanation, or formatting:
"""
    response = llm.invoke(prompt)
    score = extract_float(response.content.strip())
    if score is not None:
        return max(0.0, min(1.0, score))
    return 0.0

# 5. Compute scores for each QA pair
faithfulness_scores = []
answer_relevancy_scores = []
context_precision_scores = []
context_recall_scores = []

for i, q in enumerate(eval_questions):
    answer = ragas_answers[i]
    context = "\n\n".join(ragas_contexts[i])  # combine all retrieved chunks
    gt = ground_truths[i]

    faith = score_faithfulness(q, answer, context)
    rel = score_answer_relevancy(q, answer)
    prec = score_context_precision(q, context)
    rec = score_context_recall(gt, context)

    faithfulness_scores.append(faith)
    answer_relevancy_scores.append(rel)
    context_precision_scores.append(prec)
    context_recall_scores.append(rec)

    print(f"\n--- Question {i+1}: {q[:50]}...")
    print(f"  Faithfulness: {faith:.2f}")
    print(f"  Answer Relevancy: {rel:.2f}")
    print(f"  Context Precision: {prec:.2f}")
    print(f"  Context Recall: {rec:.2f}")

# 6. Aggregate results
avg_faith = sum(faithfulness_scores)/len(faithfulness_scores)
avg_rel = sum(answer_relevancy_scores)/len(answer_relevancy_scores)
avg_prec = sum(context_precision_scores)/len(context_precision_scores)
avg_rec = sum(context_recall_scores)/len(context_recall_scores)

print("\n=== RAGAS Evaluation Results (Manual) ===")
print(f"Faithfulness: {avg_faith:.3f}")
print(f"Answer Relevancy: {avg_rel:.3f}")
print(f"Context Precision: {avg_prec:.3f}")
print(f"Context Recall: {avg_rec:.3f}")

# 7. Per-question breakdown (DataFrame)
df_results = pd.DataFrame({
    'Question': eval_questions,
    'Faithfulness': faithfulness_scores,
    'Answer Relevancy': answer_relevancy_scores,
    'Context Precision': context_precision_scores,
    'Context Recall': context_recall_scores
})
print("\nPer-question breakdown:")
print(df_results.to_string(index=False))

# 8. NRA Insight – automatically determine weakest metric
metrics_scores = {
    'faithfulness': avg_faith,
    'answer_relevancy': avg_rel,
    'context_precision': avg_prec,
    'context_recall': avg_rec
}
weakest = min(metrics_scores, key=metrics_scores.get)

nra_t5 = f"""
Weakest metric: {weakest}
Number: {metrics_scores[weakest]:.3f}
Reason: With k=3, the retrieved contexts are short reviews that often miss dataset-level statistics required by the ground truth (e.g., aggregate percentages). The LLM's answer may also be incomplete if the context lacks key details.
Action: Increase k to 10 to capture more diverse reviews, or add a summarisation layer over the full corpus for questions needing global insights.
"""
print(nra_t5)


Q: What types of issues appear in negative freelancer reviews?
A: Based on the provided context, the types of issues that appear in negative freelancer reviews are:

- Poor quality work
...

Q: What characterises reviews where the client hired again?
A: Insufficient context....

Q: What is the approximate proportion of negative reviews in the dataset?
A: Based on the provided context, there are 3 reviews. 

1. "Quality missed feedback. poor ignored work, deadlines and." (N...

Q: What communication-related issues are mentioned in poor reviews?
A: The communication-related issues mentioned in the poor reviews are:

1. Ignored quality feedback
2. Poor communication r...

--- Question 1: What types of issues appear in negative freelancer...
  Faithfulness: 0.25
  Answer Relevancy: 0.90
  Context Precision: 0.80
  Context Recall: 0.38

--- Question 2: What characterises reviews where the client hired ...
  Faithfulness: 0.50
  Answer Relevancy: 0.00
  Context Precision: 0.20
  Context Rec

---
## Bonus — Persist & Reload + Business Summary
**[10★]**

### Task
A. Save the FAISS vector store to disk and reload it (production habit — don't re-embed every run).
B. Write a 4-sentence business summary of your RAG system as if presenting to a client.

### Skeleton

In [13]:
# Part: Bonus - Persist and Reload + Business Summary
# Goal: Save FAISS index to disk, reload it, and write a client-facing business summary.
# Method: save_local, load_local, custom summary.

# ── BONUS A: Persist FAISS index ──
vectorstore.save_local('./reviewpulse_faiss_index')
print('FAISS index saved to disk.')

vectorstore_reloaded = FAISS.load_local(
    './reviewpulse_faiss_index',
    embedding_model,
    allow_dangerous_deserialization=True
)
verify_result = vectorstore_reloaded.similarity_search('excellent professional work', k=2)
print(f'Reloaded FAISS verified. Retrieved docs: {len(verify_result)}')
for doc in verify_result:
    print(f'  [{doc.metadata["sentiment"]}] {doc.page_content}')

# ── BONUS B: Client-facing business summary ──
business_summary = """
We have built a natural language query system over 600 client reviews. The system uses 384-dimensional embeddings from all-MiniLM-L6-v2 and retrieves the 3 most relevant reviews per query from a FAISS index. Manual RAGAS evaluation gave faithfulness 0.375, answer relevancy 0.650, context precision 0.425, and context recall 0.156 – with context recall being the weakest because three short reviews miss dataset‑level patterns. Next step: increase retrieval k to 10 and add a summarisation layer over the full corpus to provide aggregate insights.
"""
print(business_summary)

FAISS index saved to disk.
Reloaded FAISS verified. Retrieved docs: 2
  [positive] Excellent very delivered professional work, on and time.!
  [positive] Professional very delivered on time. excellent work, and!

We have built a natural language query system over 600 client reviews. The system uses 384-dimensional embeddings from all-MiniLM-L6-v2 and retrieves the 3 most relevant reviews per query from a FAISS index. Manual RAGAS evaluation gave faithfulness 0.375, answer relevancy 0.650, context precision 0.425, and context recall 0.156 – with context recall being the weakest because three short reviews miss dataset‑level patterns. Next step: increase retrieval k to 10 and add a summarisation layer over the full corpus to provide aggregate insights.




---
## Scoring Rubric
| Task | Points | Criteria |
|------|--------|----------|
| T1A — Document objects | 5 | 600 docs, correct metadata keys |
| T1B — Text splitter | 5 | 600 chunks confirmed |
| T1C — NRA | 5 | Number from output, causal reason, committed action |
| T2A–B — Embedding + FAISS | 5 | 384 dims, 600 vectors |
| T2C — Similarity search | 5 | Query runs, docs printed with metadata |
| T2D — NRA | 5 | Correct semantic explanation |
| T3A–C — Chain built | 8 | Correct params: temp=0, stuff, k=3, source_docs=True |
| T3D — 4 queries run | 7 | All 4 printed with source docs |
| T3E — NRA | 5 | Specific claim, retrieval mechanism, named improvement |
| T4A — Filtered retrieval | 5 | All 3 results are negative sentiment |
| T4B — MMR retrieval | 5 | Correct params: fetch_k=10, lambda_mult=0.5 |
| T4D — NRA | 5 | lambda_mult explained correctly, committed use-case action |
| T5A–B — RAGAS dataset + eval | 8 | 4 metrics computed |
| T5C — Per-question breakdown | 3 | DataFrame printed |
| T5D — NRA | 4 | Weakest metric named, exact score, causal reason, specific action |
| **Total** | **80** | |
| Bonus A — FAISS persist + reload | 5★ | Verified with query |
| Bonus B — Business summary | 5★ | 4 sentences, ≥4 numbers, no hedging |

---
## Interview Frame
> *"I built a RAG pipeline over 600 client reviews using LangChain, FAISS, sentence-transformers, and a Groq-hosted Llama 3.1 model. The system converts reviews into 384-dimensional embeddings, indexes them with FAISS, retrieves the top-3 semantically relevant docs per query, and passes them as context to the LLM to generate grounded answers. I evaluated it with RAGAS across four metrics — faithfulness, answer relevancy, context precision, and context recall — and identified context recall as the weakest link because 3 retrieved short reviews give only partial coverage of dataset-level patterns. The fix would be increasing k or adding a summarisation layer over the full corpus before retrieval."*

---
## Month 9 — Complete Scorecard
| Day | Topic | Score |
|-----|-------|-------|
| 155 | Neural Networks & Keras | 90/90+10★ |
| 156 | CNNs | 80/80+10★ |
| 157 | RNNs & LSTMs | 80/80+10★ |
| 158 | NLP Text Processing | 80/80+10★ |
| 159 | HuggingFace Transformers | 80/80+10★ |
| 160 | RAG Pipeline (foundations) | 80/80+10★ |
| 161 | Word Embeddings | 90/90+10★ |
| 162 | Sentence Embeddings & Semantic Search | 80/80+10★ |
| 163 | Topic Modeling LDA | 80/80+10★ |
| 164 | DistilBERT Fine-Tuning | 90/90+10★ |
| 165 | Save/Load/Inference | 80/80+10★ |
| 166 | compute_metrics + Callbacks | pending |
| 167 | Multi-Class Fine-Tuning | pending |
| **168** | **RAG + LangChain Capstone** | **today** |